# FASE 3 (v2 - DIPERBARUI): YOLO26n + Coordinate Attention + Edge Contour (Google Colab GPU)
## Arsitektur Asli YOLO26n (`C3k2` + `SPPF` + `C2PSA` + `end2end=True`) + 100% Weight Remapping

Notebook ini menjalankan pipeline baru di folder **`phase3_yolo_ca/`** yang telah diperbaiki:
1. **Arsitektur Asli YOLO26n**: Menggunakan `C3k2`, `SPPF`, `C2PSA`, `reg_max=1`, dan `end2end=True`.
2. **100% Pretrained Weight Remapping**: Memetakan seluruh layer `yolo26n.pt` (`0..23` $\to$ `0..26`).
3. **Visualisasi Grafik Training & 10 Hasil Testing**: Menampilkan kurva konvergensi training HD dan grid 10 sampel prediksi Test Set.
4. **Deteksi Gambar Lapangan Interaktif**: Mendukung *upload* foto daun langsung dari HP/laptop.

### Langkah 1: Pastikan GPU Aktif & Mount Google Drive

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

### Langkah 2: Siapkan Dataset (`tea_yolo.zip`) & Kode (`phase3_yolo_ca`)

In [ ]:
import os

# 1. Ekstrak Dataset tea_yolo.zip dari Google Drive ke SSD lokal Colab (/content/)
if not os.path.exists('/content/tea_yolo'):
    !cp /content/drive/MyDrive/SKRIPSI/tea_yolo.zip /content/
    !unzip -q /content/tea_yolo.zip -d /content/
    print('[OK] Dataset tea_yolo berhasil diekstrak ke /content/tea_yolo')

# 2. Ambil kode phase3_yolo_ca & yolo26n.pt (dari Drive ZIP atau GitHub Clone)
if os.path.exists('/content/drive/MyDrive/SKRIPSI/phase3_yolo_ca.zip'):
    !cp /content/drive/MyDrive/SKRIPSI/phase3_yolo_ca.zip /content/
    !unzip -o -q /content/phase3_yolo_ca.zip -d /content/
    print('[OK] Kode phase3_yolo_ca diekstrak dari Google Drive!')
else:
    !git clone https://github.com/Kagalme/skripsi-tea-leaf-detection.git /content/repo_skripsi 2>/dev/null || (cd /content/repo_skripsi && git pull)
    !cp -r /content/repo_skripsi/phase3_yolo_ca /content/
    !cp /content/repo_skripsi/model_selection/yolo26n.pt /content/ 2>/dev/null || true
    print('[OK] Kode phase3_yolo_ca diambil dari GitHub!')

if not os.path.exists('/content/yolo26n.pt') and os.path.exists('/content/drive/MyDrive/SKRIPSI/yolo26n.pt'):
    !cp /content/drive/MyDrive/SKRIPSI/yolo26n.pt /content/

!ls -la /content/phase3_yolo_ca/
!ls -lh /content/yolo26n.pt

### Langkah 3: Install Dependency & Jalankan Unit Test Modul Baru

In [ ]:
!pip install -q ultralytics opencv-python-headless

!python /content/phase3_yolo_ca/ca_module.py
!python /content/phase3_yolo_ca/edge_contour.py

### Langkah 4: Training YOLO26n + Coordinate Attention (50 Epoch, 640×640, Batch 16)

In [ ]:
# Pastikan patch train_args={} aktif di ca_module.py sebelum training
with open('/content/phase3_yolo_ca/ca_module.py', 'r', encoding='utf-8') as f:
    _c = f.read()
if '"train_args": None' in _c:
    _c = _c.replace('"train_args": None', '"train_args": {}')
    with open('/content/phase3_yolo_ca/ca_module.py', 'w', encoding='utf-8') as f:
        f.write(_c)
    print('[FIX] train_args={} berhasil diterapkan pada ca_module.py')

!python /content/phase3_yolo_ca/01_train_ca.py \
    --device cuda \
    --epochs 50 \
    --imgsz 640 \
    --batch 16 \
    --workers 8 \
    --run_name yolo26n_ca_50ep

### Langkah 5: Evaluasi Test Set (529 Gambar, 640×640) & Visual Analysis (TP, FP, WC, FN)

In [ ]:
!python /content/phase3_yolo_ca/02_evaluate_testset.py \
    --device cuda \
    --imgsz 640 \
    --batch 16 \
    --workers 8 \
    --run_name yolo26n_ca_50ep

!python /content/phase3_yolo_ca/03_visual_analysis.py \
    --device cuda \
    --imgsz 640 \
    --run_name yolo26n_ca_50ep \
    --conf 0.25 \
    --iou_thr 0.5 \
    --max_samples 10

### Langkah 6: Tampilkan Grafik Proses Training & Evaluasi (Loss, mAP, PR Curve, Confusion Matrix)

In [ ]:
# Buat grafik proses training HD & grid 10 hasil testing
!python /content/phase3_yolo_ca/07_plot_training_and_test10.py \
    --run_name yolo26n_ca_50ep \
    --device cuda \
    --conf 0.25 \
    --num_samples 10

import os
from IPython.display import Image as IPyImage, display, Markdown

display(Markdown('#### 1. Kurva Konvergensi Proses Training (Box Loss, Cls Loss, Precision, Recall, mAP@50, F1)'))
if os.path.exists('/content/phase3_runs/yolo26n_ca_50ep/training_curves_hd.png'):
    display(IPyImage('/content/phase3_runs/yolo26n_ca_50ep/training_curves_hd.png', width=1000))

display(Markdown('#### 2. Confusion Matrix & PR Curve pada Held-Out Test Set (529 Gambar)'))
for plot_file in [
    '/content/phase3_runs/yolo26n_ca_50ep/test_eval/test_results/confusion_matrix_normalized.png',
    '/content/phase3_runs/yolo26n_ca_50ep/test_eval/test_results/BoxPR_curve.png'
]:
    if os.path.exists(plot_file):
        display(IPyImage(plot_file, width=750))

### Langkah 7: Tampilkan 10 Hasil Testing (Ground Truth vs Prediksi YOLO26n+CA + Kontur Daun)

In [ ]:
import os
from IPython.display import Image as IPyImage, display

grid10_path = '/content/phase3_runs/yolo26n_ca_50ep/10_test_predictions_grid.png'
if os.path.exists(grid10_path):
    display(IPyImage(grid10_path, width=1100))
else:
    print('Jalankan cell Langkah 6 terlebih dahulu untuk menghasilkan 10_test_predictions_grid.png')

### Langkah 8: Perbandingan Adil (640×640): YOLO26n Baseline vs YOLO26n + CA

In [ ]:
!python /content/phase3_yolo_ca/04_compare_baseline_vs_ca.py \
    --run_name_ca yolo26n_ca_50ep \
    --run_name_baseline yolo26n

from IPython.display import display, Markdown
with open('/content/laporan/fase3_ca_vs_baseline.md', 'r', encoding='utf-8') as f:
    display(Markdown(f.read()))

### Langkah 9: Upload Gambar Lapangan (Foto Kebun / Kamera HP) & Deteksi Otomatis
Jalankan cell di bawah ini untuk memilih 1 atau beberapa foto daun teh dari laptop/HP Anda. Model akan langsung mendeteksi penyakit/hama dan menampilkan perbandingan **4-Panel (Gambar Asli, BBox YOLO26n+CA, Kontur Daun Utuh, dan Peta Tepi Canny)**.

In [ ]:
import os
from google.colab import files
from IPython.display import Image as IPyImage, display, Markdown

print('Silakan pilih foto daun teh lapangan dari komputer/HP Anda:')
uploaded = files.upload()

os.makedirs('/content/hasil_uji_lapangan', exist_ok=True)
weights_pt = '/content/phase3_runs/yolo26n_ca_50ep/weights/best.pt'

for idx, filename in enumerate(uploaded.keys(), 1):
    out_sub = f'/content/hasil_uji_lapangan/img_{idx}'
    display(Markdown(f'---\n#### Hasil Deteksi Gambar Lapangan #{idx}: `{filename}`'))
    !python /content/phase3_yolo_ca/06_test_external_image.py \
        --image "{filename}" \
        --weights "{weights_pt}" \
        --conf 0.25 \
        --out "{out_sub}"
    
    result_png = f'{out_sub}/hasil_testing_lengkap.png'
    if os.path.exists(result_png):
        display(IPyImage(result_png, width=950))

### Langkah 10: Cadangkan Seluruh Hasil, Grafik, & Laporan ke Google Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/SKRIPSI/phase3_runs/yolo26n_ca_50ep
!mkdir -p /content/drive/MyDrive/SKRIPSI/phase3_runs/hasil_uji_lapangan
!mkdir -p /content/drive/MyDrive/SKRIPSI/laporan

!cp -r /content/phase3_runs/yolo26n_ca_50ep/* /content/drive/MyDrive/SKRIPSI/phase3_runs/yolo26n_ca_50ep/
!cp -r /content/hasil_uji_lapangan/* /content/drive/MyDrive/SKRIPSI/phase3_runs/hasil_uji_lapangan/ 2>/dev/null || true
!cp /content/laporan/fase3_ca_vs_baseline.md /content/drive/MyDrive/SKRIPSI/laporan/

print('[SELESAI] Semua bobot best.pt, grafik training, 10 hasil testing, hasil uji lapangan, dan laporan Fase 3 telah tersimpan di Google Drive!')